# Basic Example

This demo shows users how to generate traffic using PyBTLS. 
It generates a 1-month 1-lane Auxerre-equivalent traffic without calculating any load effect. 

To do this, we need to: 

**Step 1:** Define a traffic generator; 

**Step 2:** Request the output of vehicle files; 

**Step 3:** Set and run the simulation. 

**Step 4:** Load the generated traffic.

---
Import all the necessary modules.

In [18]:
import pybtls as pb
from pathlib import Path

---
**Step 1:** Define a traffic generator, which consists of three components:

1.1. Lane flow composition (LFC): determines the traffic flow rate and vehicle class proportions. 

1.2. Vehicle generator: determines the vehicle types and their characteristics.

1.3. Headway generator: determines the vehicle headway distribution.

---
1.1. Define the lane flow composition.
The traffic will have the following properties:
- The traffic is identical among 24 hours of a day. 
- 80 cars/hour/lane and 20 trucks/hour/lane are generated.
- The mean vehicle speed is 80 km/h with a standard deviation of 10 km/h. 
- Trucks are in the following composition: 

|      No. Axles     | 2  |  3  |  4   |  5   |
|--------------------|----|-----|------|------|
| Relative comps (%) | 23 | 2.8 | 31.7 | 42.5 |


In [ ]:
lfc = pb.LaneFlowComposition(lane_index = 1, lane_dir = 1)
lfc.assign_lane_data(  # 24 hours per day by default
    hourly_truck_flow=[80] * 24,
    hourly_car_flow=[20] * 24,
    hourly_speed_mean=[80 / 3.6 * 10] * 24,  # in dm/s
    hourly_speed_std=[10 / 3.6 * 10] * 24,  # in dm/s
    hourly_truck_composition=[[23, 2.8, 31.7, 42.5]] * 24,
)

---
1.2. Choose the Grave vehicle generator for Auxerre-equivalent traffic. 

(Note that the Grave generator is designed for several built-in traffic sites. 
It is recommended to use the Garage generator for customized traffic sites, which will be covered in later tutorials.)

In [20]:
vehicle_gen = pb.VehicleGenGrave(traffic_site = "Auxerre")

---
1.3. Choose the Freeflow headway generator (Poisson arrival theory).

In [21]:
headway_gen = pb.HeadwayGenFreeflow()

---
Getting the traffic generator by combining the lane flow composition, vehicle generator, and headway generator.

In [22]:
traffic_gen = pb.TrafficGenerator(no_lane = 1)
traffic_gen.add_lane(
    vehicle_gen = vehicle_gen,
    headway_gen = headway_gen,
    lfc = lfc,
)

---
**Step 2:** Specify the output we want - the traffic data. 

In [ ]:
output_config = pb.OutputConfig()
output_config.set_vehicle_file_output(write_vehicle_file = True)

---
**Step 3:** Set and start a new simulation.

The finished traffic output is stored in `./temp/minimum_example`. 

In [ ]:
sim_task = pb.Simulation(output_dir = Path("./temp"))  # Can specify a different output directory here. 
sim_task.add_sim(
    traffic = traffic_gen,
    no_day = 25,  # 25 working days per month
    output_config = output_config,
    tag = "minimum_example",
)
sim_task.run()

---
**Step 4:** Load the generated traffic as a dataframe. 

In [31]:
sim_output = sim_task.get_output()["minimum_example"]
generated_vehicle = sim_output.read_data("traffic")["output_traffic"]

from IPython.display import display  # Display the variable in jupyter notebook (optional)
display(generated_vehicle)

,Head,Year,Month,Day,Hour,Min,Sec,NoAxles,NoAxleGroups,GVW,Velocity,Length,Lane,Dir,Trns,AxleWeights,AxleSpacings,AxleWidths
0,1001,1,1,0,0,0,21.757,5,0,380.22579,18.888889,10.660,1,1,1.8,"[58.32045000000001, 145.02123, 58.958100000000...","[3.307, 5.243, 1.119, 0.991, 0.0]","[1.98, 1.98, 1.98, 1.98, 1.98]"
1,1001,1,1,0,0,1,0.945,2,0,144.15795,17.777778,5.932,1,1,1.8,"[46.15605000000001, 98.0019]","[5.932, 0.0]","[1.98, 1.98]"
2,1001,1,1,0,0,1,33.245,2,0,52.31673,20.555556,5.946,1,1,1.8,"[14.37165, 37.945080000000004]","[5.946, 0.0]","[1.98, 1.98]"
3,1001,1,1,0,0,1,42.168,3,0,243.16047,21.666667,6.336,1,1,1.8,"[80.61858000000001, 98.22753000000002, 64.30455]","[5.136, 1.199, 0.0]","[1.98, 1.98, 1.98]"
4,1001,1,1,0,0,1,51.885,4,0,546.17175,21.388889,10.524,1,1,1.8,"[85.41567, 187.11594000000002, 136.82007000000...","[3.157, 6.077, 1.29, 0.0]","[1.98, 1.98, 1.98, 1.98]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58635,1001,25,1,0,23,59,1.680,4,0,423.10530,25.555556,10.053,1,1,1.8,"[67.01211, 145.85508000000002, 105.12396000000...","[3.104, 6.006, 0.943, 0.0]","[1.98, 1.98, 1.98, 1.98]"
58636,1001,25,1,0,23,59,9.968,2,0,20.00259,20.277778,4.000,1,1,1.8,"[9.996390000000002, 9.996390000000002]","[4.0, 0.0]","[1.98, 1.98]"
58637,1001,25,1,0,23,59,19.765,5,0,427.84353,31.111111,11.061,1,1,1.8,"[61.06725000000001, 128.84454000000002, 79.313...","[3.315, 5.595, 1.045, 1.106, 0.0]","[1.98, 1.98, 1.98, 1.98, 1.98]"
58638,1001,25,1,0,23,59,53.692,4,0,455.04666,20.555556,11.081,1,1,1.8,"[67.78710000000001, 154.90971000000002, 116.17...","[3.267, 6.349, 1.464, 0.0]","[1.98, 1.98, 1.98, 1.98]"
